# Ensemble of Specialized Mixture of Experts (MoE) for NLI
This notebook implements a state-of-the-art Natural Language Inference pipeline combining:
1. **T5 Data Augmentation**: Generating synthetic hypotheses to improve robustness.
2. **POS-Specialized MoE**: A custom architecture with experts for Semantics, Entities, Actions, and Logic.
3. **Model Ensembling**: Averaging predictions from **DeBERTa-v3** and **ModernBERT** backbones.

In [1]:
!pip install datasets transformers torch spacy pandas numpy scikit-learn
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 28.2 MB/s eta 0:00:00m eta 0:00:010:00:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [2]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import spacy
from datasets import Dataset
from transformers import (
    AutoTokenizer, 
    AutoModel, 
    T5Tokenizer, 
    T5ForConditionalGeneration, 
    TrainingArguments, 
    Trainer
)
from sklearn.metrics import f1_score, accuracy_score

nlp = spacy.load("en_core_web_sm", disable=["ner"])
device = "cuda" if torch.cuda.is_available() else "cpu"

## 1. Data Augmentation (T5)
We use a T5 model to generate synthetic training examples to expand the diversity of our dataset.

In [3]:
def augment_dataset(df, n_samples=10):
    # The T5 model uses both encoder and decoder to generate text
    gen_model_name = "google/flan-t5-base"
    gen_tokenizer = T5Tokenizer.from_pretrained(gen_model_name)
    # ConditionalGeneration since we are generating text not just classifying (from hugging face)
    generator = T5ForConditionalGeneration.from_pretrained(gen_model_name).to(device)

    # Disables drop-out to improve model accuracy
    generator.eval()
    synthetic_examples = []
    # We only select a subset since running on the whole set is slow + we don't want to rely entirely on fake data
    subset = df.sample(n=min(n_samples, len(df)))

    print("Generating synthetic data...")
    for _, ex in subset.iterrows():
        # The context for generating the new sentence
        prompt = f"make a sentence that means this: {ex['hypothesis']}"
        inputs = gen_tokenizer(prompt, return_tensors="pt").to(device)
        outputs = generator.generate(**inputs, max_length=64)
        gen_hyp = gen_tokenizer.decode(outputs[0], skip_special_tokens=True)

        promptP = f"make a sentence that means this: {ex['premise']}"
        inputsP = gen_tokenizer(promptP, return_tensors="pt").to(device)
        outputsP = generator.generate(**inputsP, max_length=64)
        gen_prem = gen_tokenizer.decode(outputsP[0], skip_special_tokens=True)

        # Store the generated sentence in the same form as the existing data
        synthetic_examples.append({"premise": gen_prem, "hypothesis": gen_hyp, "label": ex["label"]})

    return pd.concat([df, pd.DataFrame(synthetic_examples)]).reset_index(drop=True)

# Load local data
try:
    train_df = pd.read_csv("training_data/NLI/train.csv")
    dev_df = pd.read_csv("training_data/NLI/dev.csv")
    augmented_train_df = augment_dataset(train_df)

    # augmented_train_df = train_df.head(1000).copy()
    # dev_df = dev_df.head(1000).copy()
except FileNotFoundError:
    print("CSV files not found. Please ensure training_data/train.csv exists.")

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Generating synthetic data...


In [4]:
print(augmented_train_df.tail(20))

                                                 premise  \
24422  Why do you believe apathy is present both nati...   
24423                  It's difficult to understand how.   
24424  Prawns are called gambas, while the larger var...   
24425  He bought a skate from the fish seller and cre...   
24426  According to Hall, groups offering free legal ...   
24427  Additionally, the analogy touches on what migh...   
24428  Guests have the option of dining in cozy rooms...   
24429        A lady is carrying an infant near a window.   
24430  A mattress pad, also known as a mattress toppe...   
24431                        I noticed that one as well.   
24432                the Queen Mary was launched in 1936   
24433  In this arrangement, team members offer her fe...   
24434  He does not believe in national parks (private...   
24435  Once all European prices are quoted in euros, ...   
24436  The market is located in front of the ferry te...   
24437  The hotel is located in the centr

## 2. Multi-Backbone MoE Architecture
Each model in the ensemble utilizes a Mixture of Experts layer. This layer routes features to specialized heads based on POS-filtered text (Nouns for Entities, Verbs for Actions) and manual Logic features.

In [5]:
class POSSpecializedMoE(nn.Module):
    def __init__(self, model_name, num_labels=2):
        super().__init__()
        # Force the encoder to load and stay in FP32
        self.encoder = AutoModel.from_pretrained(model_name).float() 
        hidden_size = self.encoder.config.hidden_size
        
        num_logic_features = 4
        self.semantic_expert = nn.Linear(hidden_size, hidden_size)
        self.entity_expert = nn.Linear(hidden_size, hidden_size)
        self.action_expert = nn.Linear(hidden_size, hidden_size)
        self.logic_expert = nn.Linear(num_logic_features, hidden_size)
        
        self.logic_norm = nn.LayerNorm(num_logic_features)
        self.gating = nn.Sequential(
            nn.Linear(hidden_size + num_logic_features, 128),
            nn.ReLU(),
            nn.Linear(128, 4),
            nn.Softmax(dim=-1)
        )
        
        self.classifier = nn.Linear(hidden_size, num_labels)
        self.loss_fn = nn.CrossEntropyLoss()

    def forward(self, input_ids, attention_mask, entity_ids=None, action_ids=None, logic_features=None, labels=None, **kwargs):
            # FORCE everything to Float32 immediately
            # This ensures compatibility regardless of what the Trainer tries to do
            self.encoder.float() 
            
            outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
            cls_output = outputs.last_hidden_state[:, 0, :].float() 

            if logic_features is None:
                logic_features = torch.zeros((cls_output.size(0), 4), device=cls_output.device)
            
            logic_features = logic_features.float()
            
            # Now every single operation below is guaranteed to be Float32
            norm_logic = self.logic_norm(logic_features)

            e_sem = torch.tanh(self.semantic_expert(cls_output))
            e_ent = torch.tanh(self.entity_expert(cls_output))
            e_act = torch.tanh(self.action_expert(cls_output))
            e_log = torch.tanh(self.logic_expert(norm_logic))

            gate_input = torch.cat([cls_output, norm_logic], dim=-1)
            gate_weights = self.gating(gate_input) 

            experts = torch.stack([e_sem, e_ent, e_act, e_log], dim=1)
            moe_output = torch.bmm(gate_weights.unsqueeze(1), experts).squeeze(1)

            logits = self.classifier(moe_output)

            loss = None
            if labels is not None:
                loss = self.loss_fn(logits, labels)

            return {"loss": loss, "logits": logits} if loss is not None else {"logits": logits}

## 3. Preprocessing and Feature Extraction
We extract linguistic features using Spacy to feed the MoE gating and specialized experts.

In [6]:
# Extract words based on a specific POS
def get_pos_filtered_text(text, pos_tags):
    # Generate POS tags
    doc = nlp(str(text))
    # Only select those which have been specified
    tokens = [token.text for token in doc if token.pos_ in pos_tags]
    return " ".join(tokens) if tokens else "none"

def extract_logic_features(premise, hypothesis):
    # Produce POS tags for both the premise and hypothesis
    p_doc, h_doc = nlp(str(premise)), nlp(str(hypothesis))
    # Count how many negations we see between the 2 sentences - a mismatch indicates contradiction
    neg_p = sum(1 for t in p_doc if t.dep_ == "neg")
    neg_h = sum(1 for t in h_doc if t.dep_ == "neg")
    # Removes punctuation and stopwords
    p_set = {t.lemma_.lower() for t in p_doc if not t.is_stop and not t.is_punct}
    h_set = {t.lemma_.lower() for t in h_doc if not t.is_stop and not t.is_punct}
    # Determines the overlap between sentences using Jaccard Similarity
    overlap = len(p_set & h_set) / len(p_set | h_set) if (p_set | h_set) else 0.0
    return [float(neg_p), float(neg_h), float(abs(neg_p - neg_h)), float(overlap)]

# Factory to handle HuggingFace map function
def make_preprocess_fn(tokenizer):
    def preprocess(example):
        main_enc = tokenizer(example["premise"], example["hypothesis"], truncation=True, padding="max_length", max_length=128)
        
        # Extract the relevant sentences for each expert
        p_nouns = get_pos_filtered_text(example["premise"], ["NOUN", "PROPN"])
        h_nouns = get_pos_filtered_text(example["hypothesis"], ["NOUN", "PROPN"])
        p_verbs = get_pos_filtered_text(example["premise"], ["VERB"])
        h_verbs = get_pos_filtered_text(example["hypothesis"], ["VERB"])
        
        # Encoders for the 2 experts
        ent_enc = tokenizer(p_nouns, h_nouns, truncation=True, padding="max_length", max_length=128)
        act_enc = tokenizer(p_verbs, h_verbs, truncation=True, padding="max_length", max_length=128)
        
        # Returns the sentences which each expert relies on
        return {
            "input_ids": main_enc["input_ids"],
            "attention_mask": main_enc["attention_mask"],
            "entity_ids": ent_enc["input_ids"],
            "action_ids": act_enc["input_ids"],
            "logic_features": extract_logic_features(example["premise"], example["hypothesis"]),
            "label": int(example["label"])
        }
    return preprocess

## 4. Training and Ensembling
We train two separate MoE models (DeBERTa and ModernBERT) and then ensemble their outputs by averaging the logits.

In [ ]:
from transformers import EarlyStoppingCallback, DefaultDataCollator

# Dictionary mapping backbone to its specific optimal search results
optimal_configs = {
    "deberta": {
        "lr": 3e-5, 
        "batch_size": 16, 
        "epochs": 12 # Based on your previous run's peak
    },
    "modernbert": {
        "lr": 3e-5, 
        "batch_size": 8, 
        "epochs": 8 # Based on your previous run's peak
    }
}

backbones = {
    "deberta": "microsoft/deberta-v3-small",
    "modernbert": "answerdotai/ModernBERT-base"
}

all_logits = {}

for name, path in backbones.items():
    print(f"\n--- Training Optimized MoE: {name} ---")
    
    # Setup for specific backbone
    config = optimal_configs[name]
    tokenizer = AutoTokenizer.from_pretrained(path)
    model = POSSpecializedMoE(path).to(device)
    
    prep_fn = make_preprocess_fn(tokenizer)
    train_ds = Dataset.from_pandas(augmented_train_df).map(prep_fn)
    dev_ds = Dataset.from_pandas(dev_df).map(prep_fn)
    
    args = TrainingArguments(
        output_dir=f"moe_{name}_optimized",
        learning_rate=config["lr"],
        per_device_train_batch_size=config["batch_size"],
        num_train_epochs=config["epochs"],
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True, # Rolls back to highest accuracy checkpoint
        metric_for_best_model="accuracy",
        fp16=False,
        bf16=False,
        report_to="none"
    )

    trainer = Trainer(
        model=model, 
        args=args, 
        train_dataset=train_ds, 
        eval_dataset=dev_ds,
        data_collator=DefaultDataCollator(),
        # Stops if no improvement for 3 evaluations to save time
        callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
        compute_metrics=lambda p: {"accuracy": accuracy_score(p.label_ids, np.argmax(p.predictions, axis=1))}
    )
    
    trainer.train()
    
    # Store the best predictions for the ensemble
    preds = trainer.predict(dev_ds)
    all_logits[name] = preds.predictions

# Ensemble Logic: Average Logits
final_logits = (all_logits["deberta"] + all_logits["modernbert"]) / 2
final_preds = np.argmax(final_logits, axis=1)

print("\n--- Final Optimized Ensemble Metrics ---")
print(f"Accuracy: {accuracy_score(dev_df['label'], final_preds):.4f}")
print(f"Macro F1: {f1_score(dev_df['label'], final_preds, average='macro'):.4f}")


--- Training MoE with deberta backbone ---


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Map:   0%|          | 0/24442 [00:00<?, ? examples/s]

KeyboardInterrupt: 

In [ ]:
# import itertools
# from transformers import DefaultDataCollator

# # 1. Define the search space
# param_grid = {
#     "learning_rate": [1e-5, 3e-5],
#     "per_device_train_batch_size": [8, 16],
#     "num_train_epochs": [2, 3]
# }

# # Generate all combinations of hyperparameters
# keys, values = zip(*param_grid.items())
# combinations = [dict(zip(keys, v)) for v in itertools.product(*values)]

# results_file = "hyperparameter_results.txt"

# with open(results_file, "w") as f:
#     f.write("Backbone, LR, BatchSize, Epochs, Accuracy, F1\n")

# print(f"Starting search over {len(combinations)} combinations per backbone...")

# for name, path in backbones.items():
#     tokenizer = AutoTokenizer.from_pretrained(path)
#     prep_fn = make_preprocess_fn(tokenizer)
    
#     # Pre-process datasets once per backbone to save time
#     train_ds = Dataset.from_pandas(augmented_train_df).map(prep_fn)
#     dev_ds = Dataset.from_pandas(dev_df).map(prep_fn)
    
#     for params in combinations:
#         print(f"\nTesting {name} with: {params}")
        
#         # Re-initialize model for each run to reset weights
#         model = POSSpecializedMoE(path).to(device)
        
#         args = TrainingArguments(
#             output_dir=f"search_{name}",
#             learning_rate=params["learning_rate"],
#             per_device_train_batch_size=params["per_device_train_batch_size"],
#             num_train_epochs=params["num_train_epochs"],
#             eval_strategy="no", # Speed up search by only evaluating at the end
#             save_strategy="no",
#             fp16=False,
#             bf16=False,
#             report_to="none"
#         )
        
#         trainer = Trainer(
#             model=model,
#             args=args,
#             train_dataset=train_ds,
#             data_collator=DefaultDataCollator()
#         )
        
#         trainer.train()
        
#         # Evaluate
#         preds_output = trainer.predict(dev_ds)
#         preds = np.argmax(preds_output.predictions, axis=1)
        
#         acc = accuracy_score(dev_df['label'], preds)
#         f1 = f1_score(dev_df['label'], preds, average='macro')
        
#         # Store results
#         result_line = f"{name}, {params['learning_rate']}, {params['per_device_train_batch_size']}, {params['num_train_epochs']}, {acc:.4f}, {f1:.4f}"
#         print(f"Result: {result_line}")
        
#         with open(results_file, "a") as f:
#             f.write(result_line + "\n")

# print(f"\nSearch complete. Results saved to {results_file}")

Starting search over 8 combinations per backbone...


Map:   0%|          | 0/24442 [00:00<?, ? examples/s]

Map:   0%|          | 0/6736 [00:00<?, ? examples/s]


Testing deberta with: {'learning_rate': 1e-05, 'per_device_train_batch_size': 8, 'num_train_epochs': 2}


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

...

Result: modernbert, 3e-05, 16, 2, 0.8838, 0.8836

Testing modernbert with: {'learning_rate': 3e-05, 'per_device_train_batch_size': 16, 'num_train_epochs': 3}


Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

ModernBertModel LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     |  | 
------------------+------------+--+-
head.norm.weight  | UNEXPECTED |  | 
decoder.bias      | UNEXPECTED |  | 
head.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Step,Training Loss
500,0.676971
1000,0.661181
1500,0.640315
2000,0.583167
2500,0.555424
3000,0.542580
3500,0.437562
4000,0.424641
4500,0.403022


Result: modernbert, 3e-05, 16, 3, 0.7130, 0.7130

Search complete. Results saved to hyperparameter_results.txt


In [ ]:
# from transformers import EarlyStoppingCallback, DefaultDataCollator

# # Define the optimal parameters found from search
# optimal_configs = {
#     "deberta": {"lr": 3e-5, "batch_size": 16},
#     "modernbert": {"lr": 3e-5, "batch_size": 8}
# }

# final_results_file = "final_training_results.txt"

# # Prepare the log file
# with open(final_results_file, "w") as f:
#     f.write("Backbone, Epochs_Completed, Final_Accuracy, Final_F1\n")

# all_final_logits = {}

# for name, path in backbones.items():
#     print(f"\n--- Final Long-Run Training: {name} ---")
    
#     config = optimal_configs[name]
#     tokenizer = AutoTokenizer.from_pretrained(path)
#     prep_fn = make_preprocess_fn(tokenizer)
    
#     train_ds = Dataset.from_pandas(augmented_train_df).map(prep_fn)
#     dev_ds = Dataset.from_pandas(dev_df).map(prep_fn)
    
#     model = POSSpecializedMoE(path).to(device)
    
#     training_args = TrainingArguments(
#         output_dir=f"final_model_{name}",
#         learning_rate=config["lr"],
#         per_device_train_batch_size=config["batch_size"],
#         num_train_epochs=50,
#         eval_strategy="epoch",      # Required for EarlyStopping
#         save_strategy="epoch",      # Required for EarlyStopping
#         load_best_model_at_end=True, # Ensure we keep the best version
#         metric_for_best_model="accuracy",
#         fp16=False,
#         bf16=False,
#         report_to="none",
#         logging_steps=10
#     )
    
#     trainer = Trainer(
#         model=model,
#         args=training_args,
#         train_dataset=train_ds,
#         eval_dataset=dev_ds,
#         data_collator=DefaultDataCollator(),
#         # Stopping if accuracy doesn't improve for 5 epochs
#         callbacks=[EarlyStoppingCallback(early_stopping_patience=5)],
#         compute_metrics=lambda p: {
#             "accuracy": accuracy_score(p.label_ids, np.argmax(p.predictions, axis=1))
#         }
#     )
    
#     # Train and save results
#     train_result = trainer.train()
    
#     # Final Predict
#     preds_output = trainer.predict(dev_ds)
#     all_final_logits[name] = preds_output.predictions
    
#     final_acc = accuracy_score(dev_df['label'], np.argmax(all_final_logits[name], axis=1))
#     final_f1 = f1_score(dev_df['label'], np.argmax(all_final_logits[name], axis=1), average='macro')
    
#     # Record to file
#     with open(final_results_file, "a") as f:
#         f.write(f"{name}, {train_result.global_step}, {final_acc:.4f}, {final_f1:.4f}\n")
    
#     print(f"Finished {name}. Best Accuracy: {final_acc:.4f}")

# # --- Final Ensemble Step ---
# ensemble_logits = (all_final_logits["deberta"] + all_final_logits["modernbert"]) / 2
# ensemble_preds = np.argmax(ensemble_logits, axis=1)

# print("\n--- Final Optimized Ensemble Metrics ---")
# print(f"Accuracy: {accuracy_score(dev_df['label'], ensemble_preds):.4f}")
# print(f"Macro F1: {f1_score(dev_df['label'], ensemble_preds, average='macro'):.4f}")


--- Final Long-Run Training: deberta ---


Map:   0%|          | 0/24442 [00:00<?, ? examples/s]

Map:   0%|          | 0/6736 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

Map:   0%|          | 0/6736 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

ModernBertModel LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     |  | 
------------------+------------+--+-
head.norm.weight  | UNEXPECTED |  | 
decoder.bias      | UNEXPECTED |  | 
head.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch,Training Loss,Validation Loss,Accuracy
1,0.516765,0.465685,0.790083
2,0.337350,0.327080,0.871586
3,0.158666,0.566458,0.877969
4,0.008423,0.652090,0.872031
5,0.000388,0.785302,0.870546
6,0.227751,0.836531,0.868023
7,0.097345,1.104919,0.871289
8,0.043084,1.009364,0.865350


Finished modernbert. Best Accuracy: 0.8780

--- Final Optimized Ensemble Metrics ---
Accuracy: 0.8879
Macro F1: 0.8878
